# Analyse des causes d'attrition — TechNova Partners

**Contexte** : TechNova Partners est une ESN (conseil en transfo digitale + vente de SaaS). Le département RH constate un turnover plus élevé que d'habitude et veut comprendre pourquoi. On dispose de 3 fichiers de données internes pour mener l'analyse.

**Données fournies** :
- `extrait_sirh.csv` : infos RH classiques (âge, salaire, poste, ancienneté…)
- `extrait_eval.csv` : les évaluations annuelles de performance
- `extrait_sondage.csv` : un sondage bien-être + la colonne qui nous intéresse : si l'employé a quitté l'entreprise ou non

## 1. Configuration et imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42

## 2. Chargement et exploration des données

In [ ]:
df_sirh = pd.read_csv('data/extrait_sirh.csv')
df_eval = pd.read_csv('data/extrait_eval.csv')
df_sondage = pd.read_csv('data/extrait_sondage.csv')

print(f'SIRH : {df_sirh.shape}')
print(f'Eval : {df_eval.shape}')
print(f'Sondage : {df_sondage.shape}')

In [ ]:
df_sirh.head()

In [ ]:
df_eval.head()

In [ ]:
df_sondage.head()

In [ ]:
# Vérification des types et valeurs manquantes
for name, df_tmp in [('SIRH', df_sirh), ('Eval', df_eval), ('Sondage', df_sondage)]:
    print(f'\n--- {name} ---')
    print(df_tmp.dtypes)
    missing = df_tmp.isnull().sum()
    if missing.any():
        print(f'Valeurs manquantes :\n{missing[missing > 0]}')
    else:
        print('Aucune valeur manquante')

In [ ]:
# Stats descriptives numériques
df_sirh.describe()

In [ ]:
df_eval.describe()

In [ ]:
df_sondage.describe()

## 3. Jointure des fichiers

Les 3 fichiers partagent un identifiant employé sous des formats différents :
- SIRH : `id_employee` (entier)
- Eval : `eval_number` (format `E_X`)
- Sondage : `code_sondage` (format `000000X`)

On extrait l'identifiant numérique de chaque fichier avant de les joindre.

In [ ]:
# Extraction de l'identifiant numérique
df_eval['id_employee'] = df_eval['eval_number'].str.replace('E_', '', regex=False).astype(int)
df_sondage['id_employee'] = df_sondage['code_sondage'].astype(int)

# Vérification rapide
common_ids = set(df_sirh['id_employee']) & set(df_eval['id_employee']) & set(df_sondage['id_employee'])
print(f'Employés communs aux 3 fichiers : {len(common_ids)}')
print(f'SIRH : {df_sirh.id_employee.nunique()}, Eval : {df_eval.id_employee.nunique()}, Sondage : {df_sondage.id_employee.nunique()}')

In [ ]:
# Jointure inner en deux temps
df = (
    df_sirh
    .merge(df_eval, on='id_employee', how='inner')
    .merge(df_sondage, on='id_employee', how='inner')
)

# Nettoyage des colonnes redondantes
df = df.drop(columns=['eval_number', 'code_sondage'])

print(f'DataFrame consolidé : {df.shape}')
df.head()

## 4. Nettoyage des données

In [ ]:
# La colonne augmentation contient des ' %' à retirer
df['augementation_salaire_precedente'] = (
    df['augementation_salaire_precedente']
    .str.replace(' %', '', regex=False)
    .astype(int)
)

# Vérification des doublons
print(f'Doublons : {df.duplicated().sum()}')

# Vérification des valeurs manquantes
print(f'Valeurs manquantes total : {df.isnull().sum().sum()}')

df.dtypes

In [ ]:
# Aperçu des modalités pour les colonnes catégorielles
for col in df.select_dtypes(include='object').columns:
    print(f'{col} : {df[col].unique()}')

## 5. Analyse exploratoire

On compare systématiquement les employés ayant quitté l'entreprise avec ceux encore en poste.

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['a_quitte_l_entreprise'].value_counts()
counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Effectifs')
axes[0].set_ylabel('Nombre')

counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[1].set_ylabel('')
axes[1].set_title('Proportions')

plt.suptitle("Répartition de l'attrition", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Taux d\'attrition : {(counts["Oui"] / len(df) * 100):.1f}%')

### 5.1 Variables numériques vs attrition

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.drop('id_employee')

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i >= len(axes):
        break
    sns.boxplot(data=df, x='a_quitte_l_entreprise', y=col, ax=axes[i],
                palette={'Oui': '#e74c3c', 'Non': '#2ecc71'})
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')

# Masquer les axes inutilisés
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Variables numériques selon le statut attrition', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Variables catégorielles vs attrition

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.drop('a_quitte_l_entreprise')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    if i >= len(axes):
        break
    ct = pd.crosstab(df[col], df['a_quitte_l_entreprise'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], stacked=True)
    axes[i].set_title(f'Attrition par {col}', fontsize=10)
    axes[i].set_ylabel('%')
    axes[i].legend(title='', fontsize=8)
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 5.3 Premières observations

Ce qui ressort des graphiques :
- Il y a un vrai déséquilibre dans la variable cible : la majorité des employés n'ont pas quitté l'entreprise
- Les heures sup semblent jouer un gros rôle — le taux de départ est visiblement plus élevé quand il y en a
- Les plus jeunes avec peu d'ancienneté partent plus facilement
- Le salaire médian est plus faible chez les partants, ce qui est assez logique
- Les consultants et représentants commerciaux semblent plus touchés

Tout ça reste à confirmer avec la modélisation.

## 6. Préparation des features

Objectif : obtenir un DataFrame `X` purement numérique et un vecteur cible `y` binaire.

In [ ]:
# Séparation cible / features
y = df['a_quitte_l_entreprise'].map({'Oui': 1, 'Non': 0})
X = df.drop(columns=['a_quitte_l_entreprise', 'id_employee'])

print(f'X : {X.shape}, y : {y.shape}')
print(f'Distribution cible : {y.value_counts().to_dict()}')

In [ ]:
# Encodage des variables binaires
binary_map = {
    'genre': {'F': 0, 'M': 1},
    'heure_supplementaires': {'Non': 0, 'Oui': 1},
    'ayant_enfants': {'N': 0, 'Y': 1}
}

for col, mapping in binary_map.items():
    X[col] = X[col].map(mapping)

# Encodage ordinal pour frequence_deplacement
freq_map = {'Aucun': 0, 'Occasionnel': 1, 'Frequent': 2}
X['frequence_deplacement'] = X['frequence_deplacement'].map(freq_map)

# One-Hot Encoding pour les catégorielles nominales restantes
cat_to_encode = ['departement', 'poste', 'statut_marital', 'domaine_etude']
X = pd.get_dummies(X, columns=cat_to_encode, drop_first=True)

print(f'Shape après encodage : {X.shape}')
X.head()

### 6.1 Matrice de corrélation

In [ ]:
plt.figure(figsize=(18, 14))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5)
plt.title('Matrice de corrélation (Pearson)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Paires très corrélées (|r| > 0.7)
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append((corr.columns[i], corr.columns[j], round(r, 2)))

if high_corr:
    print('Paires fortement corrélées :')
    for c1, c2, r in high_corr:
        print(f'  {c1}  <->  {c2}  : {r}')
else:
    print('Aucune paire avec |r| > 0.7')

In [ ]:
# Suppression éventuelle de colonnes redondantes
cols_to_drop = []
if high_corr:
    # On retire la 2e colonne de chaque paire (choix arbitraire mais cohérent)
    for c1, c2, r in high_corr:
        if c2 not in cols_to_drop:
            cols_to_drop.append(c2)
    print(f'Colonnes supprimées : {cols_to_drop}')
    X = X.drop(columns=cols_to_drop)
else:
    print('Rien à supprimer')

print(f'Shape final de X : {X.shape}')

### 6.2 Feature engineering

In [ ]:
# Quelques features dérivées qui peuvent aider
X['ratio_salaire_experience'] = X['revenu_mensuel'] / (X['annee_experience_totale'] + 1)
X['anciennete_vs_poste'] = X['annees_dans_l_entreprise'] - X['annees_dans_le_poste_actuel']
X['satisfaction_moyenne'] = (
    X['satisfaction_employee_environnement']
    + X['satisfaction_employee_nature_travail']
    + X['satisfaction_employee_equipe']
    + X['satisfaction_employee_equilibre_pro_perso']
) / 4

print(f'Shape avec features dérivées : {X.shape}')

In [ ]:
# Vérification : tout est numérique, pas de NaN
assert X.select_dtypes(include='object').empty, 'Il reste des colonnes non numériques !'
assert X.isnull().sum().sum() == 0, 'Il reste des valeurs manquantes !'
print('OK — X est prêt pour la modélisation')

## 7. Modélisation — Première approche

Méthodologie :
1. Split train/test stratifié (80/20)
2. Modèle Dummy (baseline)
3. Régression Logistique (modèle linéaire)
4. Random Forest (modèle non-linéaire)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Proportion cible (train) : {y_train.value_counts(normalize=True).to_dict()}')
print(f'Proportion cible (test)  : {y_test.value_counts(normalize=True).to_dict()}')

### 7.1 Modèle Dummy (baseline)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print('--- Dummy (most_frequent) ---')
print(classification_report(y_test, y_pred_dummy, target_names=['Reste', 'Part']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dummy, display_labels=['Reste', 'Part'])
plt.title('Dummy')
plt.show()

### 7.2 Régression Logistique

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print('--- Régression Logistique ---')
print(classification_report(y_test, y_pred_lr, target_names=['Reste', 'Part']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=['Reste', 'Part'])
plt.title('Régression Logistique')
plt.show()

### 7.3 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('--- Random Forest ---')
print(classification_report(y_test, y_pred_rf, target_names=['Reste', 'Part']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels=['Reste', 'Part'])
plt.title('Random Forest')
plt.show()

### 7.4 Comparaison des modèles

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def eval_model(name, y_true, y_pred):
    return {
        'Modèle': name,
        'Accuracy': round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall': round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1': round(f1_score(y_true, y_pred, zero_division=0), 3)
    }

results = pd.DataFrame([
    eval_model('Dummy', y_test, y_pred_dummy),
    eval_model('Rég. Logistique', y_test, y_pred_lr),
    eval_model('Random Forest', y_test, y_pred_rf)
]).set_index('Modèle')

results.style.highlight_max(axis=0, color='#d4edda')

Le Random Forest s'en sort mieux que le Dummy et la Logistique, c'est rassurant. Par contre le recall sur la classe \"Part\" est sans doute assez bas vu le déséquilibre. Il va falloir traiter ça.

## 8. Amélioration — Gestion du déséquilibre

Le jeu de données est déséquilibré (beaucoup plus de "Reste" que de "Part"). On teste deux approches :
- SMOTE (suréchantillonnage de la classe minoritaire)
- Validation croisée stratifiée pour des métriques plus fiables

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Avant SMOTE : {dict(pd.Series(y_train).value_counts())}')
print(f'Après SMOTE : {dict(pd.Series(y_train_res).value_counts())}')

In [ ]:
# Random Forest sur données rééquilibrées
rf_smote = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf_smote.fit(X_train_res, y_train_res)
y_pred_rf_smote = rf_smote.predict(X_test)

print('--- Random Forest + SMOTE ---')
print(classification_report(y_test, y_pred_rf_smote, target_names=['Reste', 'Part']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf_smote, display_labels=['Reste', 'Part'])
plt.title('RF + SMOTE')
plt.show()

### 8.1 XGBoost + SMOTE

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb.fit(X_train_res, y_train_res)
y_pred_xgb = xgb.predict(X_test)

print('--- XGBoost + SMOTE ---')
print(classification_report(y_test, y_pred_xgb, target_names=['Reste', 'Part']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb, display_labels=['Reste', 'Part'])
plt.title('XGBoost + SMOTE')
plt.show()

### 8.2 Validation croisée stratifiée

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models_cv = {
    'RF + SMOTE': rf_smote,
    'XGBoost + SMOTE': xgb
}

for name, model in models_cv.items():
    scores = cross_val_score(model, X_train_res, y_train_res, cv=cv, scoring='f1')
    print(f'{name:20s} — F1 moyen : {scores.mean():.3f} (± {scores.std():.3f})')

### 8.3 Synthèse comparative

In [ ]:
results_v2 = pd.DataFrame([
    eval_model('Dummy', y_test, y_pred_dummy),
    eval_model('Rég. Logistique', y_test, y_pred_lr),
    eval_model('Random Forest', y_test, y_pred_rf),
    eval_model('RF + SMOTE', y_test, y_pred_rf_smote),
    eval_model('XGBoost + SMOTE', y_test, y_pred_xgb)
]).set_index('Modèle')

results_v2.style.highlight_max(axis=0, color='#d4edda')

Avec le SMOTE le recall remonte pas mal, c'est ce qu'on voulait. XGBoost et RF donnent des résultats assez proches. On va partir sur XGBoost pour la suite et tenter de l'optimiser.

## 9. Optimisation des hyperparamètres

On lance un GridSearchCV sur XGBoost pour trouver la meilleure combinaison d'hyperparamètres. Le scoring retenu est le F1-score, plus pertinent que l'accuracy en contexte déséquilibré.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_child_weight': [1, 3, 5]
}

grid = GridSearchCV(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', use_label_encoder=False),
    param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_res, y_train_res)

print(f'Meilleurs paramètres : {grid.best_params_}')
print(f'Meilleur F1 (CV) : {grid.best_score_:.3f}')

In [ ]:
# Évaluation du modèle optimisé sur le jeu de test
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)

print('--- Modèle final (XGBoost optimisé) ---')
print(classification_report(y_test, y_pred_best, target_names=['Reste', 'Part']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best, display_labels=['Reste', 'Part'])
plt.title('Modèle final — XGBoost optimisé')
plt.show()

### 9.1 Courbe Precision-Recall

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test, name='XGBoost optimisé')
plt.title('Courbe Precision-Recall')
plt.show()

### 9.2 Détermination du seuil optimal

Par défaut le modèle utilise 0.5 comme seuil de probabilité. On va chercher le seuil qui maximise le F1-score sur le jeu de test.

In [ ]:
from sklearn.metrics import precision_recall_curve

# Probabilités prédites pour la classe 1 (Part)
y_proba = best_model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# Calcul du F1 pour chaque seuil
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'Seuil optimal : {best_threshold:.3f}')
print(f'F1 à ce seuil : {f1_scores[best_idx]:.3f}')
print(f'Precision : {precisions[best_idx]:.3f}, Recall : {recalls[best_idx]:.3f}')

In [ ]:
# Visualisation du F1 en fonction du seuil
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores, color='#2ecc71', linewidth=2)
ax.axvline(best_threshold, color='#e74c3c', linestyle='--', label=f'Seuil optimal = {best_threshold:.2f}')
ax.set_xlabel('Seuil de probabilité')
ax.set_ylabel('F1-score')
ax.set_title('F1-score en fonction du seuil de classification')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Prédictions avec le seuil optimisé
y_pred_threshold = (y_proba >= best_threshold).astype(int)

print(f'--- Avec seuil optimisé ({best_threshold:.2f}) ---')
print(classification_report(y_test, y_pred_threshold, target_names=['Reste', 'Part']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_threshold, display_labels=['Reste', 'Part'])
plt.title(f'Matrice de confusion — seuil = {best_threshold:.2f}')
plt.show()

## 10. Interprétation du modèle avec SHAP

SHAP (SHapley Additive exPlanations) permet de comprendre l'impact de chaque variable sur les prédictions, aussi bien au niveau global qu'individuel.

### 10.1 Feature importance globale — Beeswarm Plot

In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer(X_test)

shap.plots.beeswarm(shap_values, max_display=15, show=True)

### 10.2 Comparaison : Permutation Importance vs SHAP

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(best_model, X_test, y_test, n_repeats=10,
                              random_state=RANDOM_STATE, scoring='f1')

# Top 15
sorted_idx = perm.importances_mean.argsort()[-15:]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(range(len(sorted_idx)), perm.importances_mean[sorted_idx], color='#3498db')
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels(X_test.columns[sorted_idx])
ax.set_xlabel('Diminution moyenne du F1')
ax.set_title('Permutation Importance (top 15)')
plt.tight_layout()
plt.show()

Les deux approches sont globalement d'accord sur les variables qui comptent le plus, ce qui est bon signe. Les heures sup, le salaire, l'âge et la satisfaction reviennent systématiquement.

### 10.3 Feature importance locale — Waterfall Plots

In [ ]:
# Exemple 1 : un employé qui a quitté l'entreprise
idx_depart = y_test[y_test == 1].index
pos_depart = [list(X_test.index).index(i) for i in idx_depart[:1]][0]

print(f'Employé ayant quitté (index test = {pos_depart})')
shap.plots.waterfall(shap_values[pos_depart], max_display=12)

In [ ]:
# Exemple 2 : un employé resté dans l'entreprise
idx_reste = y_test[y_test == 0].index
pos_reste = [list(X_test.index).index(i) for i in idx_reste[:1]][0]

print(f'Employé resté (index test = {pos_reste})')
shap.plots.waterfall(shap_values[pos_reste], max_display=12)

### 10.4 SHAP scatter — Zoom sur les variables clés

In [ ]:
# Impact des heures supplémentaires
shap.plots.scatter(shap_values[:, 'heure_supplementaires'], color=shap_values)

In [ ]:
# Impact du revenu mensuel
shap.plots.scatter(shap_values[:, 'revenu_mensuel'], color=shap_values)

## 11. Conclusions

### Ce qu'on retient
- Le XGBoost optimisé est le meilleur modèle qu'on a obtenu. Le SMOTE a bien aidé sur le recall.
- La démarche Dummy → Logistique → RF → XGBoost permet de montrer la progression.

### Les causes principales de départ
- Les **heures sup** arrivent en tête, c'est le facteur le plus discriminant
- Un **salaire bas** pousse clairement vers la sortie
- Les **jeunes employés** avec peu d'ancienneté sont les plus à risque
- La **satisfaction** (environnement de travail, équilibre pro/perso) compte beaucoup
- La **distance domicile-travail** a aussi un impact, même si c'est moins fort

### Pistes pour les RH
- Encadrer les heures supplémentaires (surtout quand c'est systématique)
- Revoir les salaires sur les postes juniors / consultants
- Accompagner les nouveaux arrivants les 2 premières années
- Travailler sur la satisfaction (conditions de travail, flexibilité)